# 2 · Query the Vector Search Index

Runs a similarity search against the index created in notebook `1` to confirm
retrieval works before wiring it into the Supervisor Agent.

### Define Parameters

In [ ]:
dbutils.widgets.text("CATALOG_NAME", "workspace", "Catalog Name")
dbutils.widgets.text("SCHEMA_NAME", "marketing_campaign", "Schema Name")
dbutils.widgets.text("INDEX_NAME", "contract_chunks_index", "Vector Search index name")
dbutils.widgets.text("QUESTION", "What are the termination clauses in our contracts?", "Question")
dbutils.widgets.text("NUM_RESULTS", "5", "Number of results")

CATALOG_NAME = dbutils.widgets.get("CATALOG_NAME")
SCHEMA_NAME = dbutils.widgets.get("SCHEMA_NAME")
INDEX_NAME = dbutils.widgets.get("INDEX_NAME")
QUESTION = dbutils.widgets.get("QUESTION")
NUM_RESULTS = int(dbutils.widgets.get("NUM_RESULTS"))
INDEX_FQN = f"{CATALOG_NAME}.{SCHEMA_NAME}.{INDEX_NAME}"

### Similarity Search

In [ ]:
from databricks.sdk import WorkspaceClient
w = WorkspaceClient()

resp = w.vector_search_indexes.query_index(
    index_name=INDEX_FQN,
    columns=['file_name', 'chunk_index', 'content'],
    query_text=QUESTION,
    num_results=NUM_RESULTS,
)

data = (resp.result.data_array if resp.result else []) or []
print(f'Top {len(data)} matches for: {QUESTION}\n')
for row in data:
    fname, cidx, content, score = row[0], row[1], row[2], row[-1]
    print(f'[{float(score):.3f}] {fname} (chunk {cidx})')
    print('   ', str(content)[:200].replace(chr(10), ' '), '...')
    print()